<table border=0 width="100%"><tr><td><font size=5><B>week04-homework-api-data</B></font></td></tr></table>


# 第四周作业：选择1家上市公司并结合tushare获取数据进行分析
- 自由选择1家感兴趣的上市公司；
- 针对你选择的公司，提出一个你感兴趣的问题；
- 应用tushare获取与该公司相关的数据（不限于股市价格数据、财务等数据，国家宏观等数据均可使用）；
- 尝试应用你采集的数据做数据分析，回答你提出的问题。


提交作业：
- 在线提交：http://swufe.fanya.chaoxing.com/portal
- 截止日期：2026年10月16日晚上8点
- 提交**这个jupyter文档**，命名为“week04-homework-api-data_学号_姓名.ipynb”，学号姓名替换为自己的学号和姓名
- jupyter文档你要完成的包括四部分：公司名称与基本简介、研究问题题目及其内涵介绍、数据采集、数据分析与问题解答
- 为了批改方便，我这里将tushare token直接放到代码中


In [ ]:
import os
# 为了批改方便，按作业要求把 tushare token 放在代码里。采集函数只从环境变量读取。
token = "5c426f5af978fd42fe0f81fe7e60c02ada33d6c79347b61ccb44f521"
os.environ["TUSHARE_TOKEN"] = token
import tushare as ts
ts.set_token(token)
pro = ts.pro_api()


# 宝钢股份低碳承诺与具体行动的短期市场反应

第四周数据分析作业。研究对象为宝山钢铁股份有限公司（宝钢股份，600019.SH）。本 notebook 将Tushare取得的交易行情与公司报告、公开公告及可信媒体中核验的事件日期结合，做一个小样本探索性分析。

## 第一部分 公司名称与基本简介

**公司名称：宝山钢铁股份有限公司（宝钢股份，600019.SH）**

宝钢股份是上海证券交易所上市的钢铁企业，主营钢铁产品制造与销售。钢铁生产具有能源密集、碳排放强度较高的行业特征，企业的减碳目标、低碳项目建设与低碳产品商业化，可能影响投资者对其转型成本、技术能力和未来竞争力的判断。本作业以宝钢为单一案例，观察不同类型低碳信息披露之后的短期股票市场表现。

## 第二部分 研究问题题目及其内涵介绍

**题目：低碳承诺与具体项目行动的市场反应比较——基于宝钢股份公开披露事件的EDA**

**研究问题：2015—2025年经公开资料检索并核验的宝钢股份低碳信息事件中，披露具体低碳项目投资、建设、投产或低碳产品落地信息后的3个交易日市场相对收益，是否高于仅披露低碳目标、承诺或路线图的事件？观察到的组间差异是否由少数事件驱动？**

这里比较的是投资者对两类可识别信息的短期价格反应。**承诺/目标类**包括减碳目标、碳中和愿景、行动方案或路线图；**具体行动类**包括项目投资、开工、投产、绿色融资、低碳产品或商业化合作。项目开工、融资、合作和产品首发都不能直接等同于已支付全部投资或已实现减排。事件分类以公开原文为依据，关键词只用于检索候选。

“市场相对收益”定义为公告后宝钢3个交易日复权收益率的复合收益，减去同一窗口上证综指复合收益。它只扣除市场整体涨跌，未控制钢铁行业因素，也不能解释为公告导致股价变化。

## 第三部分 数据采集

本研究把数据采集分成两条可复现的工作流：**行情用 API 批量获取，低碳事件用公开资料检索后逐条核验**。配套脚本为本目录下的 `collect_data.py`；它不把网页关键词命中直接当成事件，也不会自动替研究者判断“承诺”或“行动”。

### 3.1 行情数据：TuShare API

通过 TuShare Pro 获取宝钢股份（`600019.SH`）日行情、上证综指（`000001.SH`）日行情及宝钢复权因子，日期范围为 2014-01-01 至 2025-12-31。2014 年行情用于覆盖样本首年附近的交易与复权计算。脚本读取环境变量中的 Token，不把凭据写进 notebook 或数据文件；每个接口返回后按交易日排序、去重并保存 CSV。

首次运行前安装依赖并设置 Token：

```bash
pip install tushare pandas requests beautifulsoup4
# macOS/Linux: export TUSHARE_TOKEN='你的Token'
# Windows PowerShell: $env:TUSHARE_TOKEN='你的Token'
python collect_data.py --market-only --start 20140101 --end 20251231
python collect_data.py --financial-only --start 20140101 --end 20251231
```

输出为 `data/stock_daily.csv`、`data/index_daily.csv` 和 `data/adj_factor.csv`。如果公告接口 `anns_d` 无权限，本研究不依赖它；因此不会把公告样本误称为由 TuShare 全量抓取。

### 3.2 低碳事件：公开资料检索、页面留档与人工编码

候选事件来自宝钢公司/交易所披露、公司报告、地方政府信息和可信行业/财经媒体。检索时围绕“宝钢/宝钢股份 + 低碳、减碳、氢基竖炉、绿色钢铁、项目投资、开工、投产、绿色产品”等词组合查找；纳入前核对原文或可靠报道，记录**公开日期、标题、来源 URL、支持分类的原文证据、事件类别、项目归属和同期信息干扰**。只有明确新增长期目标/路线图的列为承诺/目标类；有具体项目、融资、建设、投产或低碳产品落地内容的列为具体行动类。混合信息保留备注并进入敏感性分析。

`collect_data.py --sources-only` 会按事件表逐个请求来源网页，保存页面标题、元描述、抓取状态、最终 URL 和可读文本快照到 `source_audit/`，同时生成 `source_audit/source_audit.csv`。页面抓取用于复核和留档；网站可能限流、改版、下线或返回 PDF，因此抓取失败不等于事件不存在，自动提取也不能替代人工阅读原文。事件编码表 `事件来源与编码.csv` 记录当前纳入的13条样本及其证据。

```bash
python collect_data.py --sources-only
# 或一次运行行情采集与来源页面复核
python collect_data.py --start 20140101 --end 20251231
```

### 3.3 清洗、匹配与样本范围

事件日期按信息公开日而非项目实际施工日编码；公告日在非交易日时，以其后的交易日作为第1个交易日。行情按交易日排序，宝钢收盘价乘复权因子后计算收益；再与同日上证综指收益匹配。停牌或缺少有效交易收益的日期不填零。每条事件保留来源证据和混杂提示，之后只使用 `include=True` 且已人工复核的事件进入分析。

当前数据包包括13条已复核事件（承诺/目标类3条、具体行动类10条）以及行情快照。它们是公开资料检索得到的**目的性核验样本**，不是2015—2025年公告完整普查；2015—2020年未检索到记录不能解释为没有事件。重跑脚本可以刷新行情并检查当前来源页面，但若要扩展总体样本，仍须系统检索公告档案、补入候选事件、人工去重与编码，并留存检索日期和排除理由。

### 3.4 数据获取接口与完整采集代码

除公开网页的事件来源外，本项目的数据采集脚本还包括以下 TuShare API 接口：

| API接口 | 获取内容 | 本研究用途 |
|---|---|---|
| `daily` | 宝钢股份日行情 | 计算公告窗口个股收益 |
| `index_daily` | 上证综指日行情 | 市场基准收益 |
| `adj_factor` | 宝钢复权因子 | 处理分红送转等价格调整 |
| `daily_basic` | 换手率、市盈率、市净率等日度指标 | 补充市场背景/描述；不作为主结果 |
| `income` | 利润表 | 标记事件附近业绩披露等混杂信息 |
| `balancesheet` | 资产负债表 | 补充财务背景 |
| `cashflow` | 现金流量表 | 补充财务背景 |
| `fina_indicator` | 财务指标 | 补充盈利能力和经营表现 |
| `anns_d` | 上市公司公告索引 | 仅在显式选择时尝试；本账号此前返回权限错误 |

**主分析实际使用的是前三项行情数据和人工核验的事件表。**财务接口属于补充采集，帮助核对事件窗口内是否同时披露业绩；除非明确纳入模型并解释识别限制，不把这些财务变量冒充为主结果。公告索引权限不足时，来源检索仍按公开网页进行，不将空结果解释成“没有公告”。

下面代码单元包含 `collect_data.py` 的完整采集程序（不含命令行启动段）；运行单元只定义函数，不会自动联网。按前述命令运行脚本，或在本目录下明确调用 `fetch_tushare(...)`、`fetch_financials(...)`、`audit_event_sources()`。所有API结果保存为CSV，接口错误写入采集状态文件；Token只从 `TUSHARE_TOKEN` 环境变量读取，不嵌入代码。


In [ ]:
#!/usr/bin/env python3
"""Download market/financial data from TuShare and audit event-source pages.

The script does not discover or classify every green announcement on the web.
It downloads the market series and checks the 13 manually screened source URLs
in ``事件来源与编码.csv``. Event inclusion/category still require human review.

Usage:
    pip install tushare pandas requests beautifulsoup4
    export TUSHARE_TOKEN='your-token'
    python collect_data.py --start 20140101 --end 20251231

TuShare responses and source-page snapshots are saved under ``data/`` and
``source_audit/``. Existing files are replaced only when the command succeeds.
"""
from __future__ import annotations

import argparse
import os
import re
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

ROOT = Path.cwd()
if not (ROOT / "事件来源与编码.csv").exists():
    for candidate in [Path.cwd() / "courses" / "week04", *Path.cwd().parents]:
        if (candidate / "事件来源与编码.csv").exists():
            ROOT = candidate
            break
DATA = ROOT / "data"
AUDIT = ROOT / "source_audit"
EVENTS = ROOT / "事件来源与编码.csv"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; academic-research-source-audit/1.0)",
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.7",
}


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(path)


def fetch_tushare(start: str, end: str) -> None:
    token = os.environ.get("TUSHARE_TOKEN", "").strip()
    if not token:
        raise RuntimeError("请先设置环境变量 TUSHARE_TOKEN；脚本不会把 Token 写入文件。")
    try:
        import tushare as ts
    except ImportError as exc:
        raise RuntimeError("缺少 tushare，请运行 pip install tushare") from exc

    pro = ts.pro_api(token)
    jobs = [
        ("stock_daily.csv", "daily", {"ts_code": "600019.SH", "start_date": start, "end_date": end}),
        ("index_daily.csv", "index_daily", {"ts_code": "000001.SH", "start_date": start, "end_date": end}),
        ("adj_factor.csv", "adj_factor", {"ts_code": "600019.SH", "start_date": start, "end_date": end}),
    ]
    for filename, api_name, params in jobs:
        method = getattr(pro, api_name)
        last_error = None
        for attempt in range(1, 4):
            try:
                data = method(**params)
                if data is None or data.empty:
                    raise RuntimeError(f"{api_name} 返回空数据，请检查权限、代码和日期范围。")
                data["trade_date"] = data["trade_date"].astype(str)
                data = data.sort_values("trade_date").drop_duplicates("trade_date", keep="last")
                atomic_csv(data, DATA / filename)
                print(f"TuShare {api_name}: {len(data):,} rows -> data/{filename}")
                last_error = None
                break
            except Exception as exc:  # retry transient API/network failures
                last_error = exc
                if attempt < 3:
                    time.sleep(attempt * 2)
        if last_error:
            raise RuntimeError(f"{api_name} 获取失败：{last_error}") from last_error


def fetch_financials(start: str, end: str, probe_announcements: bool = False) -> None:
    """Fetch disclosure-date-based statements and indicators, year by year.

    Annual chunks keep each request small and make partial failures explicit.
    These statements are supplementary: event-window returns remain the main
    outcome; financial releases help flag same-day earnings confounds.
    """
    token = os.environ.get("TUSHARE_TOKEN", "").strip()
    if not token:
        raise RuntimeError("请先设置环境变量 TUSHARE_TOKEN；脚本不会把 Token 写入文件。")
    try:
        import tushare as ts
    except ImportError as exc:
        raise RuntimeError("缺少 tushare，请运行 pip install tushare") from exc
    pro = ts.pro_api(token)
    years = range(int(start[:4]), int(end[:4]) + 1)
    jobs = [
        ("daily_basic", "daily_basic.csv", "trade_date"),
        ("income", "income.csv", "ann_date"),
        ("balancesheet", "balancesheet.csv", "ann_date"),
        ("cashflow", "cashflow.csv", "ann_date"),
        ("fina_indicator", "fina_indicator.csv", "ann_date"),
    ]
    manifest = {"requested_at_utc": pd.Timestamp.now(tz="UTC").isoformat(), "interfaces": {}}
    for api_name, filename, date_field in jobs:
        frames, errors = [], []
        for year in years:
            lo, hi = f"{year}0101", f"{year}1231"
            if year == int(start[:4]):
                lo = start
            if year == int(end[:4]):
                hi = end
            params = {"ts_code": "600019.SH", "start_date": lo, "end_date": hi}
            if date_field == "ann_date":
                params = {"ts_code": "600019.SH", "ann_date": lo}
                # TuShare's ann_date filters are point dates; query each calendar
                # day is too expensive. Use reporting-period date ranges instead,
                # then retain the actual announcement dates returned by the API.
                params = {"ts_code": "600019.SH", "start_date": lo, "end_date": hi}
            last_error = None
            for attempt in range(1, 4):
                try:
                    part = getattr(pro, api_name)(**params)
                    if part is not None and not part.empty:
                        frames.append(part)
                    last_error = None
                    break
                except Exception as exc:
                    last_error = exc
                    if attempt < 3:
                        time.sleep(attempt * 2)
            if last_error:
                errors.append({"year": year, "error": str(last_error)})
        if frames:
            data = pd.concat(frames, ignore_index=True)
            keys = [c for c in ["ts_code", "ann_date", "end_date", "report_type"] if c in data.columns]
            if not keys:
                keys = list(data.columns[:1])
            data = data.drop_duplicates(keys, keep="last")
            if "ann_date" in data:
                data["ann_date"] = data["ann_date"].astype(str)
                data = data.sort_values([c for c in ["ann_date", "end_date"] if c in data])
            atomic_csv(data, DATA / filename)
            manifest["interfaces"][api_name] = {"status": "partial" if errors else "ok", "rows": len(data), "errors": errors}
            print(f"TuShare {api_name}: {len(data):,} rows -> data/{filename}")
        else:
            manifest["interfaces"][api_name] = {"status": "error", "rows": 0, "errors": errors}
            print(f"TuShare {api_name}: no rows; see financial_collection_manifest.json")

    # Announcement interface is attempted only on explicit request because the
    # current account previously returned a permission error.
    if probe_announcements:
        try:
            ann = pro.anns_d(ts_code="600019.SH", start_date=start, end_date=end)
            atomic_csv(ann, DATA / "announcements_tushare.csv")
            manifest["interfaces"]["anns_d"] = {"status": "ok", "rows": len(ann)}
        except Exception as exc:
            manifest["interfaces"]["anns_d"] = {"status": "unavailable", "error": str(exc)}
    DATA.mkdir(parents=True, exist_ok=True)
    (DATA / "financial_collection_manifest.json").write_text(
        __import__("json").dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def clean_page_text(html: bytes) -> tuple[str, str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    description = ""
    tag = soup.find("meta", attrs={"name": re.compile("description", re.I)})
    if tag:
        description = str(tag.get("content", "")).strip()
    for node in soup(["script", "style", "noscript", "svg"]):
        node.decompose()
    body = soup.get_text("\n", strip=True)
    body = re.sub(r"\n{3,}", "\n\n", body)
    return title, description, body


def audit_event_sources() -> None:
    if not EVENTS.exists():
        raise FileNotFoundError(f"找不到事件来源表：{EVENTS}")
    events = pd.read_csv(EVENTS, dtype={"ann_date": str})
    required = {"event_id", "ann_date", "title", "url", "category", "reviewed", "include", "evidence"}
    missing = required - set(events.columns)
    if missing:
        raise ValueError(f"事件来源表缺少字段：{sorted(missing)}")
    if events["event_id"].duplicated().any():
        raise ValueError("event_id 必须唯一。")
    if events["url"].isna().any() or not events["url"].astype(str).str.startswith("http").all():
        raise ValueError("每条候选事件都必须有有效的 http(s) 来源 URL。")

    AUDIT.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    session.headers.update(HEADERS)
    records = []
    for row in events.itertuples(index=False):
        result = {"event_id": row.event_id, "url": row.url, "retrieved_at_utc": pd.Timestamp.now(tz="UTC").isoformat()}
        try:
            response = session.get(row.url, timeout=(10, 30), allow_redirects=True)
            result.update({"http_status": response.status_code, "final_url": response.url,
                           "content_type": response.headers.get("Content-Type", "")})
            response.raise_for_status()
            content_type = response.headers.get("Content-Type", "").lower()
            if "pdf" in content_type or response.content[:4] == b"%PDF":
                result.update({"page_title": "", "meta_description": "", "text_file": "", "audit_status": "pdf_downloaded"})
                path = AUDIT / f"{row.event_id}.pdf"
                path.write_bytes(response.content)
                result["text_file"] = str(path.relative_to(ROOT))
            else:
                title, desc, body = clean_page_text(response.content)
                path = AUDIT / f"{row.event_id}.txt"
                path.write_text(f"URL: {response.url}\nTITLE: {title}\nDESCRIPTION: {desc}\n\n{body}\n", encoding="utf-8")
                result.update({"page_title": title, "meta_description": desc,
                               "text_file": str(path.relative_to(ROOT)), "audit_status": "fetched"})
        except Exception as exc:
            result.update({"http_status": result.get("http_status", ""), "final_url": result.get("final_url", ""),
                           "page_title": "", "meta_description": "", "text_file": "",
                           "audit_status": f"failed: {type(exc).__name__}: {exc}"})
        records.append(result)
        print(f"{row.event_id}: {result['audit_status']}")
        time.sleep(1.0)  # modest request rate for source hosts

    audit = pd.DataFrame(records)
    atomic_csv(audit, AUDIT / "source_audit.csv")
    print(f"Source audit: {len(audit)} URLs -> source_audit/source_audit.csv")
    print("网页自动提取只用于复核线索；事件日期、类别、证据和纳入决定仍须人工对照原文确认。")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--start", default="20140101", help="TuShare起始日 YYYYMMDD，默认20140101")
    parser.add_argument("--end", default="20251231", help="TuShare结束日 YYYYMMDD，默认20251231")
    parser.add_argument("--market-only", action="store_true", help="只下载TuShare行情，不请求来源网页")
    parser.add_argument("--financial-only", action="store_true", help="只下载财务报表及日度估值数据")
    parser.add_argument("--probe-announcements", action="store_true", help="额外尝试anns_d；无权限会写入状态而非伪造空样本")
    parser.add_argument("--sources-only", action="store_true", help="只复核事件表中的来源网页")
    args = parser.parse_args()
    if args.start > args.end:
        parser.error("--start 不能晚于 --end")
    if args.sources_only:
        audit_event_sources()
    elif args.market_only:
        fetch_tushare(args.start, args.end)
    elif args.financial_only:
        fetch_financials(args.start, args.end, args.probe_announcements)
    else:
        fetch_tushare(args.start, args.end)
        fetch_financials(args.start, args.end, args.probe_announcements)
        audit_event_sources()



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = ROOT / "data"
events = pd.read_csv(DATA / "events_reviewed.csv", dtype={"ann_date": str})
results = pd.read_csv(DATA / "event_results.csv")
print("事件数：", len(events))
print(events.groupby("category").size().rename(index={"commitment":"承诺/目标类", "action":"具体行动类"}))
results[["event_id", "ann_date", "category", "title", "first_post_trade_date", "third_post_trade_date", "relative_3d_percentage_points"]].round(3)


### 数据口径与来源限制

- 公告日采用信息公开日；若签约/开工发生在非交易日或稍早于新闻发布日期，按公开信息可被市场获知的日期计算，不用实际施工日倒填。
- 事件后窗口为披露日之后的第1至第3个交易日；行情为复权宝钢收盘收益，基准为上证综指。
- 部分来源为交易所公告镜像、公司报告、地方政府或财经媒体；这提高了可获得性，但不等同于完整抓取交易所全量公告。
- C03与年度报告同期披露；A01/A02属于同一氢基竖炉项目，A04/A09属于同一沙特项目先后披露；C02同时介绍路线图与行动内容。这些信息重叠风险在敏感性分析中检查。

## 第四部分 数据分析与问题解答

先按事件计算公告后3日的宝钢复合收益、上证综指复合收益及两者差值；再按承诺/目标类和具体行动类比较均值、中位数与相对收益为正的事件比例。这里的独立观测单位是公告事件，不把窗口内每日收益误当成多个独立事件。

In [ ]:
# 从行情快照独立重算事件后3个交易日的相对收益，便于检查预先计算的结果表
stock = pd.read_csv(DATA / "stock_daily.csv", dtype={"trade_date": str})
index = pd.read_csv(DATA / "index_daily.csv", dtype={"trade_date": str})
af = pd.read_csv(DATA / "adj_factor.csv", dtype={"trade_date": str})
stock = stock.merge(af[["trade_date", "adj_factor"]], on="trade_date", validate="one_to_one")
stock = stock.sort_values("trade_date")
stock["adjusted_close"] = stock["close"] * stock["adj_factor"]
stock["stock_return"] = stock["adjusted_close"].pct_change(fill_method=None)
stock["previous_stock_date"] = stock["trade_date"].shift(1)
index = index.sort_values("trade_date")
index["market_return"] = index["close"].pct_change(fill_method=None)
market = index[["trade_date", "market_return"]].merge(
    stock[["trade_date", "stock_return", "previous_stock_date"]], on="trade_date", how="left")
market["trade_date_dt"] = pd.to_datetime(market["trade_date"])
valid_prev = pd.to_datetime(market["previous_stock_date"]).eq(market["trade_date_dt"].shift(1))
market.loc[~valid_prev, "stock_return"] = np.nan
rows = []
for e in events.itertuples():
    ann = pd.to_datetime(e.ann_date)
    post = market.loc[market.trade_date_dt.gt(ann)].dropna(subset=["stock_return", "market_return"]).head(3)
    if len(post) != 3:
        continue
    r_stock = (1 + post.stock_return).prod() - 1
    r_index = (1 + post.market_return).prod() - 1
    rows.append({"event_id":e.event_id, "category":e.category, "stock_3d":r_stock,
                 "index_3d":r_index, "relative_3d_pp":(r_stock-r_index)*100})
calc = pd.DataFrame(rows)
summary = calc.groupby("category").relative_3d_pp.agg(n="count", mean="mean", median="median").round(3)
summary["positive_share"] = calc.groupby("category").relative_3d_pp.apply(lambda x:(x>0).mean()).round(3)
summary.rename(index={"commitment":"承诺/目标类", "action":"具体行动类"})

In [ ]:
pd.read_csv(DATA / "sensitivity_summary.csv").round(2)

In [ ]:
from IPython.display import display
display(results.sort_values("relative_3d_percentage_points", ascending=False)[
    ["event_id","category","ann_date","relative_3d_percentage_points","title"]
].round(2))
img = plt.imread(DATA / "event_return_comparison.png")
plt.figure(figsize=(12,5.8)); plt.imshow(img); plt.axis("off"); plt.show()

### 结果解读

在13条样本中，具体行动类的3日市场相对收益均值约为 **+0.84个百分点**，中位数约为 **+0.43个百分点**；承诺/目标类均值约为 **−1.39个百分点**，中位数约为 **−1.15个百分点**。两组均值相差约 **+2.22个百分点**，中位数相差约 **+1.58个百分点**。行动类有5/10个事件相对收益为正，承诺/目标类有1/3个为正。

这表明，在本轮核验到的事件样本中，具体行动信息后的短期相对表现总体比目标/承诺信息更好一些；但行动组内部涨跌都有，承诺组只有3条，不能据此说行动公告通常会推高股价，更不能说市场已经确认了减排成效。事件期间还可能同时出现行业、钢价、财报和其他公司新闻。

### 敏感性检查与结论边界

- 去掉与年度报告同日的C03后，行动类与承诺/目标类的均值差仍约为 **+1.23个百分点**。
- 去掉混合信息较多的C02后，均值差约为 **+2.34个百分点**；但承诺/目标组仅剩2条。
- 分别剔除同项目后续信息A02或A09，均值差约为 **+2.44** 或 **+2.57个百分点**；同时剔除A02与A09后约为 **+2.86个百分点**。若再一并剔除C03，差异约为 **+1.86个百分点**。
- 这些检查表明差异方向并非只由某一条行动公告造成，但并不能消除样本筛选、行业新闻和样本量小的问题。
- 但这不是完整公告普查，事件选择可能遗漏未检索到的信息；承诺组小且两组不平衡。窗口收益是描述性相关，不是因果效应，也没有证明项目实际减排。

### 本次作业结论

基于目前核验的13条公开事件，宝钢股份具体低碳行动类披露后的3个交易日市场相对收益，在均值和中位数上均高于承诺/目标类披露；去掉年度报告同期事件及同一项目的后续出资调整后，差异方向仍为正。这个结果为研究问题提供了一个初步的、可复核的EDA答案：**就当前样本而言，包含项目推进、融资、投产或低碳产品落地信息的公告，短期市场表现比单纯目标/路线图披露更好一些。**

结论只适用于当前样本，不能推广到全部2015—2025年公告。它也不能说明股价上涨是低碳项目导致的，或公司已完成承诺。要提升可信度，应继续核对上交所全量公告和公司新闻档案，补齐早期年份，并标记同日财报、钢价与行业重大新闻；若课程只要求小型试水，本次样本足以展示“目标表述”和“可执行行动信息”可能对应不同的短期市场定价反应。

### 复现文件

本目录中的 `data/events_reviewed.csv` 是经来源核验的事件表；`data/event_results.csv` 为逐事件窗口计算结果；`data/event_return_comparison.png` 为事件比较图。行情快照包括 `stock_daily.csv`、`index_daily.csv` 与 `adj_factor.csv`。运行本 notebook 不需要Token。